<img src='Banner-S.webp'>

<h3><b><center>Competition Brief: Monthly Lucky Draw Promotion</center></b></h3>

**Overview**

This promotion is designed to reward loyal customers through a monthly lucky draw. Each month, five (5) customers will be randomly selected to win exciting prizes.


**Promotion Period**

The competition will run on a monthly basis. Each qualifying entry within a calendar month will be eligible for that month’s draw.


**Eligibility Criteria**

To qualify for entry into the lucky draw, customers must meet all of the following requirements:
* Purchase at least three (3) products from any participating chocolate brand in a single day
* Achieve a minimum total spend of 30
* Be between the ages of 18 and 35 years

**Entry Mechanism**
* Each qualifying transaction counts as one (1) entry into the monthly draw
* Multiple entries are permitted, provided each transaction meets the qualifying criteria

**Winner Selection**
* At the end of each month, five (5) winners will be selected through a random lucky draw from all eligible entries
* The draw process will be conducted in a fair and auditable manner

**Prizes**
* Winners will receive prizes as determined by the promoter (to be defined per campaign cycle)

**Winner Notification**
* Winners will be contacted using the details provided at the time of purchase or entry
* Failure to respond within a specified timeframe may result in forfeiture and selection of an alternate winner

**General Terms**
* The promoter reserves the right to verify all entries and disqualify any participant who does not meet the eligibility criteria
* Participation in the competition constitutes acceptance of these terms and conditions

**1. Import Packages**

In [ ]:
# Data Manipulation Packages
import pandas as pd
import numpy as np
from pandasql import sqldf
from datetime import datetime, timedelta

# Directory Packages
from functools import partial
import os
from pathlib import Path
from dotenv import load_dotenv
import glob

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Package To Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

**2. Create Paths**

In [ ]:
ROOT_DIR: Path = Path().resolve().parent
Root = os.path.normpath(os.getcwd() + os.sep + os.pardir)
data_path   = Root+"/Data"

**3. Import CSV To Dataframes**

In [ ]:
dataframes = {}

for file in glob.glob(os.path.join(data_path, "*.csv")):
    name = os.path.splitext(os.path.basename(file))[0]  # file name without .csv
    dataframes[name] = pd.read_csv(file)

**4. Join All Dataframes**

In [ ]:
df = pd.merge(dataframes['sales'], dataframes['products'],on='product_id', how='left')
df = pd.merge(df, dataframes['customers'],on='customer_id', how='left')
df = pd.merge(df, dataframes['stores'],on='store_id', how='left')

**5. Clean Data**

In [ ]:
df['order_month'] = df['order_date'].str[:7]
df['order_year'] = df['order_date'].str[:4]

**6. Query Data Using SQL To Get Qualifying Players**

In [ ]:
qualified = """with requirements AS 
                (SELECT customer_id
                        ,order_year
                        ,order_date
                        ,order_month
                        ,quantity
                        ,unit_price
                        ,revenue
                        ,profit
                        ,age
                        ,CASE WHEN quantity >= 3 THEN 1 ELSE 0 END AS MetQuantityReq
                        ,CASE WHEN revenue >= 30 THEN 1 ELSE 0 END AS MetRevenueReq
                        ,CASE WHEN age BETWEEN 18 AND 35 THEN 1 ELSE 0 END AS MetAgeReq
                FROM df
                WHERE loyalty_member = 1
                )
        SELECT * FROM requirements 
        WHERE order_year = '2024'
        AND MetQuantityReq = 1
        AND MetRevenueReq = 1
        AND MetAgeReq = 1
        """

df_qual = sqldf(qualified)

**7. Plot Number Of Qualifying Players**

In [ ]:
bar = df_qual.groupby('order_month')['customer_id'].count().reset_index().sort_values(by='order_month')

fig = make_subplots(specs=[[{"secondary_y": True}]])

Legend = dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='right', x=0.6)
x_tick_val = bar['order_month']

fig.add_trace(
    go.Bar(x=bar['order_month'], y=bar['customer_id'], name="Customers",marker=dict(color='blue')),
    secondary_y=False)

fig.update_layout(width=1200,height=600, title_text='<b>Qualifying Players</b>'
                  ,font_family='arial'
                  ,font_color='#202020'
                  ,title_x=0.5
                  ,font_size=15
                  ,plot_bgcolor='white'
                  ,paper_bgcolor='white'
                  ,legend=Legend
                  ,barmode='stack')

fig.update_traces(marker_line_width = 0)
fig.update_yaxes(title_text='<b>Quantity</b>',showgrid=False,range=[0, max(bar['customer_id']) + 10], secondary_y=False)
fig.update_xaxes(title_text='<b>Month </b>', tickvals=x_tick_val, type="category", tickformat='%d-%b-%Y', showgrid=False)
fig.show()

**8. Draw 5 Random Customers For Each Month & Export To CSV**

In [ ]:
def pick_winners(df, seed=42):
    df = df[['order_month','customer_id','order_date','quantity','revenue','profit','age']]
    X = df.groupby('order_month', group_keys=False).apply(lambda x: x.sample(n=5, random_state=seed)).reset_index(drop=True).sort_values(['order_month'])
    return X

winners = pick_winners(df_qual)

In [ ]:
winners.to_csv('Winning Players.csv',index=False)